In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Imports ===
import json
import random
import shutil
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import torch
from ultralytics import YOLO

import ipywidgets as widgets
from IPython.display import display, clear_output

print(f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()} | "
      f"device count: {torch.cuda.device_count()}")

In [ ]:
# === Cell 3: Config ===
PROJECT_NAME = "20260824_YOLO_Generalizability_Benchmark"

PROJECT_ROOT   = Path(f"{KI_SORT_ROOT}/KI_Sort_Project")
DATASETS_ROOT  = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets")
OUTPUT_DIR     = PROJECT_ROOT / "08_Generalizability"
RESULTS_DIR    = OUTPUT_DIR / PROJECT_NAME
STAGE_DIR      = RESULTS_DIR / "_val_stage"
VAL_RUNS_DIR   = RESULTS_DIR / "_ultralytics_val"

# Deployment-tier weights (trained on all KI-Sort data).
MODEL_PATH = (PROJECT_ROOT / "02_Image_Segmentation" / "20260603_Yolo_Deployment_Training"
              / "runs" / "20260603_Yolo_Deployment_Training_DeploymentSegTraining"
              / "weights" / "best.pt")

CLASS_MAP   = {"body": 0, "cap": 1, "label": 2}
CLASS_NAMES = {v: k for k, v in CLASS_MAP.items()}
REGIONS     = ["body", "cap", "label"]

# Label sets and provenance files inside each yolo_dataset/ folder.
TRUE_LABELS_DIRNAME   = "labels_true_gt_valid"
PSEUDO_LABELS_DIRNAME = "labels_pseudo_gt"
EXPORT_INDEX_NAME     = "export_index.csv"
TRUE_GT_META_NAME     = "true_gt_meta.csv"

# export_index.csv column holding the object stem. None triggers stem-overlap detection.
INDEX_KEY_COL = None

# Datasets with no completed Label Studio pass are reported and skipped.
SKIP_DATASETS_WITHOUT_TRUE_GT = True

# body_valid = body AND NOT union(label). YOLO-seg polygons cannot encode holes, so the
# rule is applied in the mask domain to ground truth and predictions alike. Ultralytics
# val consumes the polygon files directly and is therefore unaffected by this flag.
APPLY_BODY_VALID = True

# Deployment inference config.
CONF_THRESHOLD = 0.50
IOU_THRESHOLD  = 0.50          # NMS IoU
IMAGE_SIZE     = 640
IMAGE_RECT     = True
MAX_DET        = 300
RETINA_MASKS   = True

MATCH_IOU_THRESHOLD = 0.50     # greedy per-class instance matching threshold

# Ultralytics val is run twice per dataset: a low-confidence pass for the mAP
# integrals and a CONF_THRESHOLD pass for precision / recall / F1.
VAL_CONF_MAP = 0.001
VAL_CONF_PR  = CONF_THRESHOLD

DEVICE       = 0
RANDOM_SEED  = 42
LOG_EVERY    = 25              # ETA log interval in images
ALLOW_OVERWRITE = True

# Per-dataset paths. RUN_ID = None resolves to the newest run under <root>/data/.
# images_dir / labels_dir may be overridden with any filtered subset directory.
DATASET_SPECS = {
    "spectralwaste": {
        "display":            "SpectralWaste",
        "root":               DATASETS_ROOT / "Spectral_Waste"      / "KI_Sort_Preprocessing",
        "run_id":             None,
        "images_dir":         None,
        "labels_dir":         None,
        "pseudo_labels_dir":  None,
        "index_csv":          None,
        "meta_csv":           None,
        "color":              "#1f77b4",
    },
    "warp": {
        "display":            "WaRP",
        "root":               DATASETS_ROOT / "WaRP_Dataset"        / "KI_Sort_Preprocessing",
        "run_id":             None,
        "images_dir":         None,
        "labels_dir":         None,
        "pseudo_labels_dir":  None,
        "index_csv":          None,
        "meta_csv":           None,
        "color":              "#ff7f0e",
    },
    "packwise": {
        "display":            "PackWISE",
        "root":               DATASETS_ROOT / "PackWISE_Dataset_v2" / "KI_Sort_Preprocessing",
        "run_id":             None,
        "images_dir":         None,
        "labels_dir":         None,
        "pseudo_labels_dir":  None,
        "index_csv":          None,
        "meta_csv":           None,
        "color":              "#2ca02c",
    },
}

DATASET_COLORS = {k: v["color"] for k, v in DATASET_SPECS.items()}
POOLED_COLOR   = "#7f7f7f"

IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg")

# Metadata columns carried from export_index.csv into the per-image CSV.
META_COLUMNS = ["source", "class_name", "split", "packwise_split", "session",
                "stage1_confidence"]

# Plot style.
_FW         = 16 / 2.54
_FONT_SIZE  = 10
_BAND_COLOR = "#f0f0f0"
C_BODY, C_CAP, C_LABEL = "#FF0000", "#00FF00", "#0000FF"
REGION_COLORS = {"body": C_BODY, "cap": C_CAP, "label": C_LABEL}

# Preview widget settings (not bound to the thesis plot style).
WIDGET_FIG_W          = 17.0       # inches; height follows the image aspect
WIDGET_TITLE_PAD_IN   = 0.9
WIDGET_MASK_ALPHA     = 0.35
WIDGET_BOX_THICKNESS  = 2
WIDGET_CONTOUR_THICK  = 2
WIDGET_TEXT_SCALE     = 0.55
WIDGET_TEXT_THICKNESS = 2
WIDGET_GT_DASH_PX     = 12
WIDGET_GT_GAP_PX      = 7
WIDGET_PSEUDO_DASH_PX = 5
WIDGET_PSEUDO_GAP_PX  = 6

PRED_COLORS_RGB   = {0: (255,   0,   0), 1: (  0, 200,   0), 2: (  0,  80, 255)}
GT_COLORS_RGB     = {0: (255, 160, 160), 1: (160, 255, 160), 2: (160, 200, 255)}
PSEUDO_COLORS_RGB = {0: (255, 140,   0), 1: (180, 255,   0), 2: (  0, 200, 255)}
DEFAULT_PRED_RGB   = (255, 255,   0)
DEFAULT_GT_RGB     = (255, 255, 160)
DEFAULT_PSEUDO_RGB = (255, 255, 255)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
STAGE_DIR.mkdir(parents=True, exist_ok=True)
VAL_RUNS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"RESULTS_DIR : {RESULTS_DIR}")
print(f"MODEL_PATH  : {MODEL_PATH}  exists={MODEL_PATH.exists()}")
print(f"label set   : {TRUE_LABELS_DIRNAME}  |  body_valid applied: {APPLY_BODY_VALID}")

In [ ]:
# === Cell 4: Helpers ===
def _run_has_images(run_dir):
    images_dir = run_dir / "yolo_dataset" / "images"
    return images_dir.is_dir() and any(
        p.suffix.lower() in IMAGE_EXTENSIONS for p in images_dir.iterdir())


def resolve_run_dir(spec):
    """Return the run directory holding the yolo_dataset folder for a dataset spec.

    When run_id is None, picks the newest run whose yolo_dataset/images is populated;
    newer runs without exported images are skipped.
    """
    if spec["images_dir"] is not None:
        return None
    data_root = Path(spec["root"]) / "data"
    if not data_root.is_dir():
        raise FileNotFoundError(f"missing data root: {data_root}")
    run_id = spec["run_id"]
    if run_id is None:
        candidates = sorted([p for p in data_root.iterdir()
                             if p.is_dir() and (p / "yolo_dataset").is_dir()])
        if not candidates:
            raise FileNotFoundError(f"no run with a yolo_dataset under {data_root}")
        runs = [p for p in candidates if _run_has_images(p)]
        if not runs:
            raise FileNotFoundError(
                f"no run under {data_root} has a populated yolo_dataset/images "
                f"(newest candidate: {candidates[-1].name})")
        empty_newer = [p.name for p in candidates if p not in runs
                       and p.name > runs[-1].name]
        if empty_newer:
            print(f"note: skipping empty/unpopulated run(s) newer than "
                  f"{runs[-1].name}: {empty_newer}")
        run_dir = runs[-1]
    else:
        run_dir = data_root / run_id
    return run_dir


def resolve_dataset(key, spec):
    """Fill in the true-GT, pseudo-GT, index and meta paths for one dataset."""
    resolved = dict(spec)
    run_dir = resolve_run_dir(spec)
    if run_dir is not None:
        yolo_root = run_dir / "yolo_dataset"
        resolved["run_id"]    = run_dir.name
        resolved["yolo_root"] = yolo_root
        resolved["images_dir"]        = yolo_root / "images"
        resolved["labels_dir"]        = yolo_root / TRUE_LABELS_DIRNAME
        resolved["pseudo_labels_dir"] = yolo_root / PSEUDO_LABELS_DIRNAME
        if resolved["index_csv"] is None:
            # The true-GT status columns are written into the copy inside yolo_dataset/.
            candidates = [yolo_root / EXPORT_INDEX_NAME,
                          Path(spec["root"]) / "results" / run_dir.name / "tables"
                          / EXPORT_INDEX_NAME]
            resolved["index_csv"] = next((p for p in candidates if p.is_file()),
                                         candidates[0])
        if resolved["meta_csv"] is None:
            resolved["meta_csv"] = yolo_root / TRUE_GT_META_NAME
    else:
        resolved["yolo_root"] = Path(resolved["images_dir"]).parent
    for field in ("images_dir", "labels_dir", "pseudo_labels_dir", "index_csv", "meta_csv"):
        if resolved.get(field) is not None:
            resolved[field] = Path(resolved[field])
    resolved["key"] = key
    return resolved


def list_images(images_dir):
    d = Path(images_dir)
    if not d.is_dir():
        return []
    return sorted([p for p in d.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS])


def image_path_for(images_dir, stem):
    for ext in IMAGE_EXTENSIONS:
        p = Path(images_dir) / f"{stem}{ext}"
        if p.is_file():
            return p
    return None


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in ("true", "1", "yes", "t")


def as_str_series(series):
    """Coerce a column to plain strings, mapping every missing value to an empty string."""
    return series.map(lambda v: "" if pd.isna(v) else str(v))


def detect_index_key(idx_df, stems, override=None):
    """Return the export_index.csv column whose values match the most object stems."""
    if override:
        if override not in idx_df.columns:
            raise KeyError(f"INDEX_KEY_COL {override!r} not in {EXPORT_INDEX_NAME}")
        return override
    best, best_hits = None, -1
    for col in idx_df.columns:
        hits = sum(1 for v in as_str_series(idx_df[col])
                   if v and (v in stems or Path(v).stem in stems))
        if hits > best_hits:
            best, best_hits = col, hits
    return best if best_hits > 0 else None


def _stems_from_column(series, stems):
    return as_str_series(series).apply(lambda v: v if v in stems else Path(v).stem)


def load_true_gt_state(resolved):
    """Resolve accepted / rejected / unannotated objects for one dataset.

    Status is read from export_index.csv when the Route B columns are present, and from
    true_gt_meta.csv otherwise. Accepted stems are additionally required to have both an
    image and a labels_true_gt file on disk.
    """
    state = {
        "available": False, "source": None, "key_col": None,
        "n_index_rows": 0, "n_images": 0,
        "status_counts": {}, "reasons": {},
        "accepted": [], "rejected": [], "unannotated": [],
        "missing_label_file": [], "index": None, "notes": [],
    }
    image_stems = {p.stem for p in list_images(resolved["images_dir"])}
    state["n_images"] = len(image_stems)

    status_by_stem, reason_by_stem = {}, {}

    idx_path = resolved.get("index_csv")
    if idx_path is not None and Path(idx_path).is_file():
        idx = pd.read_csv(idx_path)
        state["n_index_rows"] = len(idx)
        if "true_gt_status" in idx.columns or "true_gt_keep" in idx.columns:
            key_col = detect_index_key(idx, image_stems, INDEX_KEY_COL)
            if key_col is None:
                state["notes"].append(f"no {EXPORT_INDEX_NAME} column matches any stem")
            else:
                idx = idx.assign(_stem=_stems_from_column(idx[key_col], image_stems))
                state["key_col"] = key_col
                state["index"] = idx
                state["source"] = f"{EXPORT_INDEX_NAME} [{key_col}]"
                statuses = (as_str_series(idx["true_gt_status"])
                            if "true_gt_status" in idx.columns
                            else idx["true_gt_keep"].apply(
                                lambda v: "accepted" if as_bool(v) else "rejected"))
                reasons = (as_str_series(idx["true_gt_reason"])
                           if "true_gt_reason" in idx.columns
                           else pd.Series([""] * len(idx), index=idx.index))
                for stem, status, reason in zip(idx["_stem"], statuses, reasons):
                    status_by_stem[stem] = status.strip() or "unannotated"
                    reason_by_stem[stem] = reason.strip()
    else:
        state["notes"].append(f"{EXPORT_INDEX_NAME} not found at {idx_path}")

    meta_path = resolved.get("meta_csv")
    if not status_by_stem and meta_path is not None and Path(meta_path).is_file():
        meta = pd.read_csv(meta_path)
        state["source"] = TRUE_GT_META_NAME
        stems_col   = as_str_series(meta["stem"]) if "stem" in meta.columns else []
        status_col  = as_str_series(meta["status"]) if "status" in meta.columns else []
        reason_col  = as_str_series(meta["reason"]) if "reason" in meta.columns else []
        for i, stem in enumerate(stems_col):
            stem = stem.strip()
            if not stem:
                continue
            status_by_stem[stem] = (status_col.iloc[i].strip()
                                    if len(status_col) else "") or "unannotated"
            reason_by_stem[stem] = reason_col.iloc[i].strip() if len(reason_col) else ""
        if state["n_index_rows"]:
            for stem in image_stems - set(status_by_stem):
                status_by_stem[stem] = "unannotated"

    if not status_by_stem:
        state["notes"].append("no true-GT status found; Route B has not been run")
        return state

    for stem, status in status_by_stem.items():
        bucket = status if status in ("accepted", "rejected") else "unannotated"
        state[bucket].append(stem)
    for bucket in ("accepted", "rejected", "unannotated"):
        state[bucket] = sorted(state[bucket])

    true_dir = Path(resolved["labels_dir"])
    usable, missing = [], []
    for stem in state["accepted"]:
        if stem in image_stems and (true_dir / f"{stem}.txt").is_file():
            usable.append(stem)
        else:
            missing.append(stem)
    state["missing_label_file"] = missing
    state["accepted"] = usable
    if missing:
        state["notes"].append(f"{len(missing)} accepted objects lack an image or a "
                              f"{TRUE_LABELS_DIRNAME} file and are dropped")

    state["status_counts"] = dict(Counter(status_by_stem.values()))
    state["reasons"] = dict(Counter(
        reason_by_stem.get(s, "") or "unspecified" for s in state["rejected"]))
    state["available"] = bool(usable)
    return state


def load_index_metadata(state):
    """Return a stem to metadata dict built from the export_index.csv columns."""
    lookup = {}
    idx = state.get("index")
    if idx is None:
        return lookup
    keep = [c for c in META_COLUMNS if c in idx.columns]
    if not keep:
        return lookup
    for _, row in idx.iterrows():
        lookup.setdefault(str(row["_stem"]), {c: row[c] for c in keep})
    return lookup


def parse_yolo_seg(txt_path, width, height):
    """Parse a YOLO polygon segmentation file into (class_id, pixel polygon) pairs."""
    items = []
    path = Path(txt_path)
    if not path.is_file():
        return items
    for line in path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) < 7:
            continue
        cls = int(float(parts[0]))
        coords = np.asarray(parts[1:], dtype=np.float64)
        if coords.size % 2:
            coords = coords[:-1]
        poly = coords.reshape(-1, 2) * np.array([width, height], dtype=np.float64)
        if poly.shape[0] < 3:
            continue
        items.append((cls, poly))
    return items


def polygon_to_mask(poly, width, height):
    mask = np.zeros((height, width), dtype=np.uint8)
    pts = np.round(np.asarray(poly, dtype=np.float64)).astype(np.int32).reshape(-1, 1, 2)
    cv2.fillPoly(mask, [pts], 1)
    return mask.astype(bool)


def instance_masks(items, width, height, apply_body_valid=None):
    """Rasterise (class_id, polygon) items into per-region lists of instance masks.

    With apply_body_valid the union of all label masks is subtracted from every body
    mask; body instances left empty by the subtraction are dropped.
    """
    if apply_body_valid is None:
        apply_body_valid = APPLY_BODY_VALID
    masks = {r: [] for r in REGIONS}
    for cls, poly in items:
        region = CLASS_NAMES.get(int(cls))
        if region is None:
            continue
        masks[region].append(polygon_to_mask(poly, width, height))
    if apply_body_valid and masks["body"] and masks["label"]:
        label_union = np.zeros((height, width), dtype=bool)
        for m in masks["label"]:
            label_union |= m
        kept = []
        for m in masks["body"]:
            valid = m & ~label_union
            if valid.any():
                kept.append(valid)
        masks["body"] = kept
    return masks


def semantic_masks(items, width, height, apply_body_valid=None):
    """Rasterise items into one unioned binary mask per region."""
    if apply_body_valid is None:
        apply_body_valid = APPLY_BODY_VALID
    out = {r: np.zeros((height, width), dtype=bool) for r in REGIONS}
    for cls, poly in items:
        region = CLASS_NAMES.get(int(cls))
        if region is None:
            continue
        out[region] |= polygon_to_mask(poly, width, height)
    if apply_body_valid:
        out["body"] = out["body"] & ~out["label"]
    return out


def predict_polygons(result):
    """Return (class_id, confidence, pixel polygon) per predicted instance."""
    items = []
    if result.masks is None or result.boxes is None:
        return items
    classes = result.boxes.cls.cpu().numpy().astype(int)
    confs   = result.boxes.conf.cpu().numpy().astype(float)
    for cls, conf, poly in zip(classes, confs, result.masks.xy):
        poly = np.asarray(poly, dtype=np.float64)
        if poly.shape[0] < 3:
            continue
        items.append((int(cls), float(conf), poly))
    return items


def mask_iou(a, b):
    union = np.logical_or(a, b).sum()
    if union == 0:
        return 0.0
    return float(np.logical_and(a, b).sum() / union)


def greedy_match(gt_masks, pred_masks):
    """Greedy highest-IoU one-to-one matching within a single class."""
    if not gt_masks or not pred_masks:
        return []
    iou = np.zeros((len(gt_masks), len(pred_masks)), dtype=np.float64)
    for i, g in enumerate(gt_masks):
        for j, p in enumerate(pred_masks):
            iou[i, j] = mask_iou(g, p)
    pairs, used_g, used_p = [], set(), set()
    order = np.argsort(iou, axis=None)[::-1]
    for flat in order:
        i, j = np.unravel_index(flat, iou.shape)
        if iou[i, j] <= 0.0:
            break
        if i in used_g or j in used_p:
            continue
        used_g.add(int(i)); used_p.add(int(j))
        pairs.append((int(i), int(j), float(iou[i, j])))
    return pairs


def safe_div(num, den):
    return float(num) / float(den) if den else 0.0


def f1_from_pr(p, r):
    return safe_div(2.0 * p * r, p + r)


def stage_dataset(resolved, stems):
    """Build a staging dir of per-object image and true-GT label symlinks plus data.yaml."""
    stage = STAGE_DIR / resolved["key"]
    if stage.exists():
        shutil.rmtree(stage)
    (stage / "images").mkdir(parents=True)
    (stage / "labels").mkdir(parents=True)
    for stem in stems:
        img = image_path_for(resolved["images_dir"], stem)
        lbl = Path(resolved["labels_dir"]) / f"{stem}.txt"
        if img is None or not lbl.is_file():
            continue
        (stage / "images" / img.name).symlink_to(img.resolve())
        (stage / "labels" / lbl.name).symlink_to(lbl.resolve())
    yaml_path = stage / "data.yaml"
    names_block = "\n".join(f"  {i}: {CLASS_NAMES[i]}" for i in sorted(CLASS_NAMES))
    # train: is required by check_det_dataset and points at the same images; it is
    # never read because validation is always invoked with split="val".
    yaml_path.write_text(
        f"path: {stage.resolve()}\n"
        f"train: images\n"
        f"val: images\n"
        f"nc: {len(CLASS_NAMES)}\n"
        f"names:\n{names_block}\n",
        encoding="utf-8")
    return yaml_path


class ProgressLogger:
    """Per-item progress with elapsed time, rate and ETA."""

    def __init__(self, total, label, every=LOG_EVERY):
        self.total = int(total)
        self.label = label
        self.every = max(1, int(every))
        self.done = 0
        self.t0 = time.time()

    def update(self, n=1):
        self.done += n
        if self.done % self.every and self.done != self.total:
            return
        elapsed = time.time() - self.t0
        rate = safe_div(self.done, elapsed)
        eta = safe_div(self.total - self.done, rate) if rate else float("nan")
        print(f"[{self.label}] {self.done}/{self.total} "
              f"({100.0 * self.done / max(self.total, 1):5.1f}%) | "
              f"elapsed {elapsed:7.1f}s | {rate:5.2f} img/s | ETA {eta:7.1f}s", flush=True)

    def finish(self):
        elapsed = time.time() - self.t0
        print(f"[{self.label}] done: {self.done} images in {elapsed:.1f}s "
              f"({safe_div(self.done, elapsed):.2f} img/s)", flush=True)
        return elapsed


def resolve_device(device):
    if isinstance(device, str) and device.lower() == "cpu":
        return "cpu"
    if not torch.cuda.is_available():
        print("CUDA unavailable — falling back to CPU.")
        return "cpu"
    if isinstance(device, int) and device >= torch.cuda.device_count():
        print(f"device {device} out of range ({torch.cuda.device_count()} visible) — using 0.")
        return 0
    return device


def load_model(model_path, device):
    model = YOLO(str(model_path))
    names = {int(k): str(v) for k, v in model.names.items()}
    for name, idx in CLASS_MAP.items():
        if names.get(idx) != name:
            print(f"WARNING: class id {idx} is '{names.get(idx)}' in the weights, "
                  f"expected '{name}'")
    print(f"model: {Path(model_path).name} | names: {names} | device: {device}")
    return model


DATASETS  = {k: resolve_dataset(k, v) for k, v in DATASET_SPECS.items()}
TRUE_GT   = {k: load_true_gt_state(v) for k, v in DATASETS.items()}
ACTIVE_DATASETS = {k: v for k, v in DATASETS.items()
                   if TRUE_GT[k]["available"] or not SKIP_DATASETS_WITHOUT_TRUE_GT}

for key, res in DATASETS.items():
    st = TRUE_GT[key]
    print(f"{res['display']:<14} run_id={res.get('run_id')}  images={st['n_images']:>4}  "
          f"accepted={len(st['accepted']):>4}  rejected={len(st['rejected']):>3}  "
          f"unannotated={len(st['unannotated']):>4}  source={st['source']}")
    for note in st["notes"]:
        print(f"{'':<14} note: {note}")

print(f"\nactive datasets: {[DATASETS[k]['display'] for k in ACTIVE_DATASETS]}")
if not ACTIVE_DATASETS:
    print("no dataset has a completed true-GT pass — run Route B before benchmarking")

In [ ]:
# === Cell 5: True-GT Provenance and Pseudo-GT Agreement ===
# Per-dataset drop statistics and per-region pseudo-GT vs true-GT IoU. Text output only;
# the collected rows are saved by the Save Result Tables cell.
AGREEMENT_MIN_PX = 1           # a region counts as present above this pixel area


def region_agreement(pseudo_masks, true_masks):
    """Per-region agreement between two semantic mask dicts of one object."""
    out = {}
    for region in REGIONS:
        p, t = pseudo_masks[region], true_masks[region]
        p_px, t_px = int(p.sum()), int(t.sum())
        inter = int(np.logical_and(p, t).sum())
        union = int(np.logical_or(p, t).sum())
        present_p = p_px >= AGREEMENT_MIN_PX
        present_t = t_px >= AGREEMENT_MIN_PX
        out[region] = {
            "present_pseudo": present_p, "present_true": present_t,
            "inter": inter, "union": union,
            "iou": (float(inter) / union) if union else np.nan,
        }
    return out


def pseudo_true_agreement(resolved, stems):
    """Accumulate per-region pseudo-GT vs true-GT agreement over the accepted objects."""
    acc = {r: {"iou": [], "both": 0, "pseudo_only": 0, "true_only": 0,
               "neither": 0, "inter": 0, "union": 0} for r in REGIONS}
    n_objects, n_missing_pseudo = 0, 0
    for stem in stems:
        img_path = image_path_for(resolved["images_dir"], stem)
        if img_path is None:
            continue
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        pseudo_txt = Path(resolved["pseudo_labels_dir"]) / f"{stem}.txt"
        true_txt   = Path(resolved["labels_dir"]) / f"{stem}.txt"
        if not pseudo_txt.is_file():
            n_missing_pseudo += 1
            continue
        p_masks = semantic_masks(parse_yolo_seg(pseudo_txt, w, h), w, h)
        t_masks = semantic_masks(parse_yolo_seg(true_txt, w, h), w, h)
        stats = region_agreement(p_masks, t_masks)
        n_objects += 1
        for region, s in stats.items():
            a = acc[region]
            a["inter"] += s["inter"]
            a["union"] += s["union"]
            if s["present_pseudo"] and s["present_true"]:
                a["both"] += 1
                a["iou"].append(s["iou"])
            elif s["present_pseudo"]:
                a["pseudo_only"] += 1
                a["iou"].append(0.0)
            elif s["present_true"]:
                a["true_only"] += 1
                a["iou"].append(0.0)
            else:
                a["neither"] += 1
    return acc, n_objects, n_missing_pseudo


def merge_accumulators(target, source):
    for region in REGIONS:
        for field in ("both", "pseudo_only", "true_only", "neither", "inter", "union"):
            target[region][field] += source[region][field]
        target[region]["iou"].extend(source[region]["iou"])
    return target


def print_agreement_block(acc, n_objects, indent="  "):
    header = (f"{indent}{'region':<7}{'n':>6}{'mean IoU':>11}{'median':>9}"
              f"{'both':>7}{'SAM3 only':>11}{'added':>8}{'absent':>8}")
    print(header)
    print(indent + "-" * (len(header) - len(indent)))
    macro = []
    for region in REGIONS:
        a = acc[region]
        vals = np.asarray(a["iou"], dtype=float)
        n = vals.size
        mean = float(np.mean(vals)) if n else np.nan
        median = float(np.median(vals)) if n else np.nan
        macro.extend(vals.tolist())
        mean_s = f"{mean:.3f}" if n else "n/a"
        med_s  = f"{median:.3f}" if n else "n/a"
        print(f"{indent}{region:<7}{n:>6}{mean_s:>11}{med_s:>9}"
              f"{a['both']:>7}{a['pseudo_only']:>11}{a['true_only']:>8}{a['neither']:>8}")
    inter = sum(acc[r]["inter"] for r in REGIONS)
    union = sum(acc[r]["union"] for r in REGIONS)
    macro_mean = float(np.mean(macro)) if macro else np.nan
    micro = (float(inter) / union) if union else np.nan
    macro_s = f"{macro_mean:.3f}" if macro else "n/a"
    micro_s = f"{micro:.3f}" if union else "n/a"
    print(f"{indent}objects compared      : {n_objects}")
    print(f"{indent}overall IoU macro     : {macro_s}   "
          f"mean over per-object per-region IoU")
    print(f"{indent}overall IoU micro     : {micro_s}   "
          f"pooled intersection over pooled union")
    return macro_mean, micro


def agreement_rows(dataset_key, display, acc, n_objects):
    """Flatten one accumulator into per-region rows plus one overall row."""
    rows, macro = [], []
    for region in REGIONS:
        a = acc[region]
        vals = np.asarray(a["iou"], dtype=float)
        macro.extend(vals.tolist())
        rows.append({
            "dataset": dataset_key, "display": display, "region": region,
            "n_objects": n_objects, "n_iou_values": int(vals.size),
            "mean_iou": float(np.mean(vals)) if vals.size else np.nan,
            "median_iou": float(np.median(vals)) if vals.size else np.nan,
            "n_both_present": a["both"], "n_pseudo_only": a["pseudo_only"],
            "n_true_only": a["true_only"], "n_absent_both": a["neither"],
            "intersection_px": a["inter"], "union_px": a["union"],
            "iou_micro": (float(a["inter"]) / a["union"]) if a["union"] else np.nan,
        })
    inter = sum(acc[r]["inter"] for r in REGIONS)
    union = sum(acc[r]["union"] for r in REGIONS)
    rows.append({
        "dataset": dataset_key, "display": display, "region": "overall",
        "n_objects": n_objects, "n_iou_values": len(macro),
        "mean_iou": float(np.mean(macro)) if macro else np.nan,
        "median_iou": float(np.median(macro)) if macro else np.nan,
        "n_both_present": sum(acc[r]["both"] for r in REGIONS),
        "n_pseudo_only": sum(acc[r]["pseudo_only"] for r in REGIONS),
        "n_true_only": sum(acc[r]["true_only"] for r in REGIONS),
        "n_absent_both": sum(acc[r]["neither"] for r in REGIONS),
        "intersection_px": inter, "union_px": union,
        "iou_micro": (float(inter) / union) if union else np.nan,
    })
    return rows


def drop_rows(dataset_key, resolved, state):
    """Flatten one dataset's true-GT status counts into long-format drop rows."""
    counts = state["status_counts"]
    n_total = sum(counts.values())
    base = {"dataset": dataset_key, "display": resolved["display"],
            "run_id": resolved.get("run_id"), "status_source": state["source"],
            "n_index_objects": n_total}
    entries = [("accepted", "", counts.get("accepted", 0)),
               ("unannotated", "", counts.get("unannotated", 0))]
    for reason, n in sorted(state["reasons"].items(), key=lambda kv: -kv[1]):
        entries.append(("rejected", reason, n))
    entries.append(("missing_label_file", "", len(state["missing_label_file"])))
    entries.append(("dropped_total", "", n_total - counts.get("accepted", 0)))
    return [{**base, "bucket": bucket, "reason": reason, "n": int(n),
             "share_of_index": (float(n) / n_total) if n_total else np.nan}
            for bucket, reason, n in entries]


DROP_ROWS, AGREEMENT_ROWS = [], []

pooled_acc = {r: {"iou": [], "both": 0, "pseudo_only": 0, "true_only": 0,
                  "neither": 0, "inter": 0, "union": 0} for r in REGIONS}
pooled_objects = 0

print("=" * 78)
print("TRUE GROUND TRUTH PROVENANCE — dropped objects per dataset")
print("=" * 78)

for key, resolved in DATASETS.items():
    state = TRUE_GT[key]
    counts = state["status_counts"]
    n_total = sum(counts.values())
    print(f"\n{resolved['display']}  (run {resolved.get('run_id')})")
    print(f"  status source        : {state['source']}")
    if not counts:
        print("  no Route B pass on disk — dataset excluded from the benchmark")
        continue
    DROP_ROWS.extend(drop_rows(key, resolved, state))
    print(f"  objects in index     : {n_total}")
    for status in ("accepted", "rejected", "unannotated"):
        n = counts.get(status, 0)
        print(f"    {status:<18} : {n:>5}  ({100.0 * n / max(n_total, 1):5.1f}%)")
    dropped = n_total - counts.get("accepted", 0)
    print(f"  dropped from benchmark: {dropped}  "
          f"({100.0 * dropped / max(n_total, 1):.1f}%)")
    if state["reasons"]:
        print("  rejection reasons:")
        for reason, n in sorted(state["reasons"].items(), key=lambda kv: -kv[1]):
            print(f"    {n:>4}  {reason}")
    other = []
    if counts.get("unannotated", 0):
        other.append((counts["unannotated"], "unannotated — skipped or never opened"))
    if state["missing_label_file"]:
        other.append((len(state["missing_label_file"]),
                      f"accepted but no {TRUE_LABELS_DIRNAME} file on disk"))
    if other:
        print("  other drops:")
        for n, note in other:
            print(f"    {n:>4}  {note}")

print("\n" + "=" * 78)
print("PSEUDO GROUND TRUTH VS TRUE GROUND TRUTH AGREEMENT — accepted objects only")
print("=" * 78)
print(f"semantic per-region IoU, rasterised at the image resolution, "
      f"body_valid applied: {APPLY_BODY_VALID}")

for key, resolved in DATASETS.items():
    state = TRUE_GT[key]
    if not state["available"]:
        continue
    acc, n_objects, n_missing = pseudo_true_agreement(resolved, state["accepted"])
    print(f"\n{resolved['display']}")
    if n_missing:
        print(f"  {n_missing} accepted objects have no {PSEUDO_LABELS_DIRNAME} file "
              f"and are excluded from the comparison")
    print_agreement_block(acc, n_objects)
    AGREEMENT_ROWS.extend(agreement_rows(key, resolved["display"], acc, n_objects))
    merge_accumulators(pooled_acc, acc)
    pooled_objects += n_objects

if pooled_objects:
    print("\nPooled across datasets")
    print_agreement_block(pooled_acc, pooled_objects)
    AGREEMENT_ROWS.extend(agreement_rows("pooled", "Pooled", pooled_acc, pooled_objects))
else:
    print("\nno accepted objects available for the pseudo-GT comparison")

print(f"\ncollected {len(DROP_ROWS)} drop rows and {len(AGREEMENT_ROWS)} agreement rows "
      f"for Cell 7")

In [ ]:
# === Cell 6: Batch Inference and Metrics ===
DEVICE_RESOLVED = resolve_device(DEVICE)
model = load_model(MODEL_PATH, DEVICE_RESOLVED)

if not ACTIVE_DATASETS:
    raise RuntimeError("no dataset has accepted true-GT objects; run Route B first")


def val_metrics(data_yaml, conf, run_name):
    """Run Ultralytics segmentation validation and extract box and mask metrics."""
    res = model.val(
        data=str(data_yaml), split="val", conf=conf, iou=IOU_THRESHOLD,
        imgsz=IMAGE_SIZE, rect=IMAGE_RECT, max_det=MAX_DET, retina_masks=RETINA_MASKS,
        device=DEVICE_RESOLVED, plots=False, save_json=False, verbose=False,
        project=str(VAL_RUNS_DIR), name=run_name, exist_ok=True,
    )
    out = {}
    present = [int(c) for c in np.atleast_1d(res.ap_class_index)]
    for task in ("box", "seg"):
        m = getattr(res, task)
        block = {
            "map50":     float(m.map50),
            "map50_95":  float(m.map),
            "precision": float(m.mp),
            "recall":    float(m.mr),
            "f1":        f1_from_pr(float(m.mp), float(m.mr)),
            "per_class": {},
        }
        for pos, cls_id in enumerate(present):
            name = CLASS_NAMES.get(cls_id, str(cls_id))
            p = float(np.atleast_1d(m.p)[pos])
            r = float(np.atleast_1d(m.r)[pos])
            block["per_class"][name] = {
                "map50":     float(np.atleast_1d(m.ap50)[pos]),
                "map50_95":  float(np.atleast_1d(m.ap)[pos].mean()
                                   if np.atleast_1d(m.ap).ndim > 1
                                   else np.atleast_1d(m.ap)[pos]),
                "precision": p,
                "recall":    r,
                "f1":        f1_from_pr(p, r),
            }
        out[task] = block
    return out


def per_image_pass(resolved, stems, meta_lookup):
    """Predict every accepted object and score greedy per-class instance IoU."""
    rows = []
    log = ProgressLogger(len(stems), resolved["display"])
    for stem in stems:
        img_path = image_path_for(resolved["images_dir"], stem)
        img = cv2.imread(str(img_path)) if img_path is not None else None
        if img is None:
            print(f"WARNING: unreadable image skipped: {stem}")
            continue
        h, w = img.shape[:2]
        gt_items = parse_yolo_seg(Path(resolved["labels_dir"]) / f"{stem}.txt", w, h)
        result = model.predict(
            source=img, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD, imgsz=IMAGE_SIZE,
            max_det=MAX_DET, retina_masks=RETINA_MASKS, device=DEVICE_RESOLVED,
            verbose=False, save=False)[0]
        pred_items = predict_polygons(result)

        gt_masks_by_region = instance_masks(gt_items, w, h)
        pr_masks_by_region = instance_masks(
            [(c, p) for c, _, p in pred_items], w, h)
        conf_by_region = {r: [] for r in REGIONS}
        for cls, conf, _ in pred_items:
            region = CLASS_NAMES.get(int(cls))
            if region is not None:
                conf_by_region[region].append(conf)

        meta = meta_lookup.get(stem, {})
        row = {
            "dataset":    resolved["key"],
            "display":    resolved["display"],
            "cutout_id":  stem,
            "image_file": img_path.name,
            "width":      w,
            "height":     h,
        }
        for col in META_COLUMNS:
            row[col] = meta.get(col, np.nan)

        for region in REGIONS:
            gt_masks = gt_masks_by_region[region]
            pr_masks = pr_masks_by_region[region]
            pr_conf  = conf_by_region[region]
            pairs = greedy_match(gt_masks, pr_masks)
            ious_matched = [iou for _, _, iou in pairs if iou >= MATCH_IOU_THRESHOLD]
            ious_all_gt = [0.0] * len(gt_masks)
            for gi, _, iou in pairs:
                ious_all_gt[gi] = iou
            tp = len(ious_matched)
            row[f"{region}_n_gt"]          = len(gt_masks)
            row[f"{region}_n_pred"]        = len(pr_masks)
            row[f"{region}_tp"]            = tp
            row[f"{region}_fp"]            = len(pr_masks) - tp
            row[f"{region}_fn"]            = len(gt_masks) - tp
            row[f"{region}_mean_iou_matched"] = float(np.mean(ious_matched)) if tp else np.nan
            row[f"{region}_mean_iou_all_gt"]  = (float(np.mean(ious_all_gt))
                                                 if gt_masks else np.nan)
            row[f"{region}_max_iou"]       = float(max(ious_all_gt)) if gt_masks else np.nan
            row[f"{region}_mean_conf"]     = float(np.mean(pr_conf)) if pr_conf else np.nan
        rows.append(row)
        log.update()
    elapsed = log.finish()
    return pd.DataFrame(rows), elapsed


def matching_summary(df):
    """Aggregate the greedy-matching counters into per-region and micro totals."""
    out = {"per_region": {}}
    tp_all = fp_all = fn_all = 0
    for region in REGIONS:
        tp = int(df[f"{region}_tp"].sum())
        fp = int(df[f"{region}_fp"].sum())
        fn = int(df[f"{region}_fn"].sum())
        p, r = safe_div(tp, tp + fp), safe_div(tp, tp + fn)
        out["per_region"][region] = {
            "tp": tp, "fp": fp, "fn": fn,
            "n_gt": int(df[f"{region}_n_gt"].sum()),
            "n_pred": int(df[f"{region}_n_pred"].sum()),
            "precision": p, "recall": r, "f1": f1_from_pr(p, r),
            "mean_iou_matched": float(df[f"{region}_mean_iou_matched"].mean(skipna=True))
                                 if df[f"{region}_mean_iou_matched"].notna().any() else 0.0,
            "mean_iou_all_gt": float(df[f"{region}_mean_iou_all_gt"].mean(skipna=True))
                                if df[f"{region}_mean_iou_all_gt"].notna().any() else 0.0,
        }
        tp_all += tp; fp_all += fp; fn_all += fn
    p, r = safe_div(tp_all, tp_all + fp_all), safe_div(tp_all, tp_all + fn_all)
    out["micro"] = {"tp": tp_all, "fp": fp_all, "fn": fn_all,
                    "precision": p, "recall": r, "f1": f1_from_pr(p, r)}
    return out


RUN_PARAMS = {
    "weights": str(MODEL_PATH),
    "tier": "deployment",
    "label_set": TRUE_LABELS_DIRNAME,
    "conf_threshold": CONF_THRESHOLD,
    "nms_iou": IOU_THRESHOLD,
    "match_iou": MATCH_IOU_THRESHOLD,
    "imgsz": IMAGE_SIZE,
    "rect": IMAGE_RECT,
    "max_det": MAX_DET,
    "retina_masks": RETINA_MASKS,
    "val_conf_map": VAL_CONF_MAP,
    "val_conf_pr": VAL_CONF_PR,
    "apply_body_valid": APPLY_BODY_VALID,
    "body_valid_scope": "greedy_matching_pass_only",
    "device": str(DEVICE_RESOLVED),
    "seed": RANDOM_SEED,
    "class_map": CLASS_MAP,
}

# Remove result files of inactive datasets so the plotting cell does not pick them up.
for key in DATASETS:
    if key in ACTIVE_DATASETS:
        continue
    for stale in (RESULTS_DIR / f"{PROJECT_NAME}_{key}_metrics.json",
                  RESULTS_DIR / f"{PROJECT_NAME}_{key}_per_image.csv"):
        if stale.exists():
            stale.unlink()
            print(f"removed stale result file: {stale.name}")

all_metrics, all_frames = {}, []
for key, resolved in ACTIVE_DATASETS.items():
    state = TRUE_GT[key]
    stems = state["accepted"]
    metrics_path  = RESULTS_DIR / f"{PROJECT_NAME}_{key}_metrics.json"
    perimage_path = RESULTS_DIR / f"{PROJECT_NAME}_{key}_per_image.csv"
    if not ALLOW_OVERWRITE and (metrics_path.exists() or perimage_path.exists()):
        raise RuntimeError(f"outputs already exist for '{key}' and ALLOW_OVERWRITE is False")

    print(f"\n=== {resolved['display']} === {len(stems)} accepted objects", flush=True)
    data_yaml = stage_dataset(resolved, stems)

    print("Ultralytics val — mAP pass", flush=True)
    map_block = val_metrics(data_yaml, VAL_CONF_MAP, f"{key}_map")
    print("Ultralytics val — precision/recall pass", flush=True)
    pr_block = val_metrics(data_yaml, VAL_CONF_PR, f"{key}_pr")

    df, elapsed = per_image_pass(resolved, stems, load_index_metadata(state))
    df.to_csv(perimage_path, index=False)
    all_frames.append(df)

    metrics = {
        "dataset": key,
        "display": resolved["display"],
        "run_id": resolved.get("run_id"),
        "images_dir": str(resolved["images_dir"]),
        "labels_dir": str(resolved["labels_dir"]),
        "pseudo_labels_dir": str(resolved["pseudo_labels_dir"]),
        "index_csv": str(resolved["index_csv"]) if resolved["index_csv"] else None,
        "n_images": int(len(df)),
        "true_gt": {
            "status_source": state["source"],
            "n_accepted": len(state["accepted"]),
            "n_rejected": len(state["rejected"]),
            "n_unannotated": len(state["unannotated"]),
            "rejection_reasons": state["reasons"],
        },
        "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S"),
        "inference_seconds": float(elapsed),
        "params": RUN_PARAMS,
        "ultralytics": {
            "map_pass": {"conf": VAL_CONF_MAP, **map_block},
            "pr_pass":  {"conf": VAL_CONF_PR,  **pr_block},
        },
        "greedy_matching": matching_summary(df),
    }
    metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    all_metrics[key] = metrics

    seg_map = map_block["seg"]
    seg_pr = pr_block["seg"]
    print(f"mask mAP50 {seg_map['map50']:.3f} | mAP50-95 {seg_map['map50_95']:.3f} | "
          f"P {seg_pr['precision']:.3f} | R {seg_pr['recall']:.3f} | "
          f"F1 {seg_pr['f1']:.3f}", flush=True)

# Pooled: image-count weighted for the mAP integrals, micro-summed for the matching counters.
weights_n = {k: m["n_images"] for k, m in all_metrics.items()}
total_n = sum(weights_n.values())


def weighted(task, block, field):
    return safe_div(sum(all_metrics[k]["ultralytics"][block][task][field] * weights_n[k]
                        for k in all_metrics), total_n)


pooled = {
    "aggregation": {"map_metrics": "image_count_weighted_mean",
                    "matching_metrics": "micro_sum"},
    "n_images": int(total_n),
    "per_dataset_n_images": weights_n,
    "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S"),
    "params": RUN_PARAMS,
    "ultralytics": {},
}
for task in ("box", "seg"):
    p = weighted(task, "pr_pass", "precision")
    r = weighted(task, "pr_pass", "recall")
    pooled["ultralytics"][task] = {
        "map50":     weighted(task, "map_pass", "map50"),
        "map50_95":  weighted(task, "map_pass", "map50_95"),
        "precision": p, "recall": r, "f1": f1_from_pr(p, r),
    }

pooled_df = pd.concat(all_frames, ignore_index=True)
pooled["greedy_matching"] = matching_summary(pooled_df)
pooled_df.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_per_image_all.csv", index=False)
(RESULTS_DIR / f"{PROJECT_NAME}_aggregate_metrics.json").write_text(
    json.dumps(pooled, indent=2), encoding="utf-8")

summary = pd.DataFrame([{
    "Dataset": m["display"],
    "Images": m["n_images"],
    "mAP50": m["ultralytics"]["map_pass"]["seg"]["map50"],
    "mAP50-95": m["ultralytics"]["map_pass"]["seg"]["map50_95"],
    "Precision": m["ultralytics"]["pr_pass"]["seg"]["precision"],
    "Recall": m["ultralytics"]["pr_pass"]["seg"]["recall"],
    "F1": m["ultralytics"]["pr_pass"]["seg"]["f1"],
    "Micro F1 (Match)": m["greedy_matching"]["micro"]["f1"],
} for m in all_metrics.values()])
print("\n" + summary.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\nwritten to {RESULTS_DIR}")

In [ ]:
# === Cell 7: Save Result Tables ===
# Writes three CSVs into RESULTS_DIR (overwritten on every run). The metrics table is built
# from the JSONs on disk; the agreement and drop tables need the in-memory rows of the
# True-GT Provenance and Pseudo-GT Agreement cell from the current session.
METRICS_CSV   = RESULTS_DIR / f"{PROJECT_NAME}_metrics_summary.csv"
AGREEMENT_CSV = RESULTS_DIR / f"{PROJECT_NAME}_pseudo_vs_true_gt_agreement.csv"
DROP_CSV      = RESULTS_DIR / f"{PROJECT_NAME}_drop_statistics.csv"

SAVE_TASK = "seg"
SAVE_METRIC_FIELDS = [("mAP50", "map50", "map_pass"),
                      ("mAP50-95", "map50_95", "map_pass"),
                      ("Precision", "precision", "pr_pass"),
                      ("Recall", "recall", "pr_pass"),
                      ("F1", "f1", "pr_pass")]


def _metric_value(m, field, block):
    """Read one metric, tolerating the pooled JSON's block-free ultralytics layout."""
    ultra = m["ultralytics"]
    if block in ultra:
        return float(ultra[block][SAVE_TASK][field])
    return float(ultra[SAVE_TASK][field])


def _per_class_value(m, region, field, block):
    ultra = m["ultralytics"]
    if block not in ultra:
        return np.nan
    per_class = ultra[block][SAVE_TASK].get("per_class", {})
    if region not in per_class:
        return np.nan
    return float(per_class[region][field])


def _metrics_row(dataset_key, display, m):
    params = m.get("params", {})
    true_gt = m.get("true_gt", {})
    row = {
        "dataset": dataset_key,
        "display": display,
        "run_id": m.get("run_id", ""),
        "metrics_timestamp": m.get("timestamp", ""),
        "n_images": m.get("n_images"),
        "label_set": params.get("label_set", ""),
        "weights": Path(params.get("weights", "")).name,
        "conf_threshold": params.get("conf_threshold"),
        "match_iou": params.get("match_iou"),
        "apply_body_valid": params.get("apply_body_valid"),
        "n_accepted": true_gt.get("n_accepted"),
        "n_rejected": true_gt.get("n_rejected"),
        "n_unannotated": true_gt.get("n_unannotated"),
    }
    for name, field, block in SAVE_METRIC_FIELDS:
        row[name] = _metric_value(m, field, block)
    row["Micro F1 (Match)"] = float(m["greedy_matching"]["micro"]["f1"])
    for region in REGIONS:
        row[f"{region}_mAP50-95"] = _per_class_value(m, region, "map50_95", "map_pass")
        row[f"{region}_F1"] = _per_class_value(m, region, "f1", "pr_pass")
        gm = m["greedy_matching"]["per_region"].get(region, {})
        row[f"{region}_match_f1"] = gm.get("f1", np.nan)
        row[f"{region}_mean_iou_all_gt"] = gm.get("mean_iou_all_gt", np.nan)
    return row


metrics_by_ds = {}
for path in sorted(RESULTS_DIR.glob(f"{PROJECT_NAME}_*_metrics.json")):
    if path.name.endswith("_aggregate_metrics.json"):
        continue
    data = json.loads(path.read_text(encoding="utf-8"))
    metrics_by_ds[data["dataset"]] = data
agg_path = RESULTS_DIR / f"{PROJECT_NAME}_aggregate_metrics.json"
aggregate = json.loads(agg_path.read_text(encoding="utf-8")) if agg_path.is_file() else None

if not metrics_by_ds:
    print(f"no metrics JSON under {RESULTS_DIR} — run Cell 6 first; metrics table skipped")
else:
    rows = [_metrics_row(k, m["display"], m) for k, m in metrics_by_ds.items()]
    if aggregate is not None:
        rows.append(_metrics_row("pooled", "Pooled", aggregate))
    metrics_df = pd.DataFrame(rows)
    metrics_df.to_csv(METRICS_CSV, index=False)
    print(f"metrics   : {METRICS_CSV.name}  ({len(metrics_df)} rows)")
    print(metrics_df[["display", "n_images"] + [n for n, _, _ in SAVE_METRIC_FIELDS]
                     + ["Micro F1 (Match)"]].to_string(
        index=False, float_format=lambda v: f"{v:.3f}"))

if not AGREEMENT_ROWS:
    print("\nno agreement rows in memory — run Cell 5; agreement table skipped")
else:
    agreement_df = pd.DataFrame(AGREEMENT_ROWS)
    agreement_df.to_csv(AGREEMENT_CSV, index=False)
    print(f"\nagreement : {AGREEMENT_CSV.name}  ({len(agreement_df)} rows)")
    print(agreement_df[["display", "region", "n_iou_values", "mean_iou", "median_iou",
                        "iou_micro"]].to_string(
        index=False, float_format=lambda v: f"{v:.3f}"))

if not DROP_ROWS:
    print("\nno drop rows in memory — run Cell 5; drop table skipped")
else:
    drop_df = pd.DataFrame(DROP_ROWS)
    drop_df.to_csv(DROP_CSV, index=False)
    print(f"\ndrops     : {DROP_CSV.name}  ({len(drop_df)} rows)")
    print(drop_df[["display", "bucket", "reason", "n", "share_of_index"]].to_string(
        index=False, float_format=lambda v: f"{v:.3f}"))

print(f"\nall tables written to {RESULTS_DIR}")

In [ ]:
# === Cell 8: Preview Widget ===
def _rgb2bgr(c):
    return (c[2], c[1], c[0])


def _pred_bgr(cid):
    return _rgb2bgr(PRED_COLORS_RGB.get(int(cid), DEFAULT_PRED_RGB))


def _gt_bgr(cid):
    return _rgb2bgr(GT_COLORS_RGB.get(int(cid), DEFAULT_GT_RGB))


def _pseudo_bgr(cid):
    return _rgb2bgr(PSEUDO_COLORS_RGB.get(int(cid), DEFAULT_PSEUDO_RGB))


def draw_dashed_polygon(img, points, color, thickness=2, dash_px=12, gap_px=7):
    """Draw a closed dashed polygon on img in-place (OpenCV has no dashed line)."""
    pts = [tuple(p) for p in points]
    if len(pts) < 2:
        return
    pts.append(pts[0])
    for i in range(len(pts) - 1):
        x0, y0 = int(pts[i][0]),     int(pts[i][1])
        x1, y1 = int(pts[i + 1][0]), int(pts[i + 1][1])
        dx, dy = x1 - x0, y1 - y0
        seg_len = max(1, int(np.hypot(dx, dy)))
        step = dash_px + gap_px
        t = 0
        while t < seg_len:
            t_end = min(t + dash_px, seg_len)
            sx = int(round(x0 + dx * t     / seg_len))
            sy = int(round(y0 + dy * t     / seg_len))
            ex = int(round(x0 + dx * t_end / seg_len))
            ey = int(round(y0 + dy * t_end / seg_len))
            cv2.line(img, (sx, sy), (ex, ey), color, thickness, cv2.LINE_AA)
            t += step


def draw_label_chip(img, anchor_xy, text, color_bgr):
    """Draw a filled single-line tag anchored above a detection."""
    x, y = int(anchor_xy[0]), int(anchor_xy[1])
    (tw, th), bl = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX,
                                   WIDGET_TEXT_SCALE, WIDGET_TEXT_THICKNESS)
    box_w, box_h = tw + 8, th + bl + 6
    top = max(0, y - box_h)
    cv2.rectangle(img, (x, top), (x + box_w, top + box_h), color_bgr, -1)
    cv2.putText(img, text, (x + 4, top + th + 3), cv2.FONT_HERSHEY_SIMPLEX,
                WIDGET_TEXT_SCALE, (255, 255, 255), WIDGET_TEXT_THICKNESS, cv2.LINE_AA)


def preview_predict(resolved, img_path, conf, iou):
    """Run the model on one object and collect the prediction, true-GT and pseudo-GT items."""
    img_bgr = cv2.imread(str(img_path))
    h, w = img_bgr.shape[:2]
    stem = Path(img_path).stem
    gt_items     = parse_yolo_seg(Path(resolved["labels_dir"]) / f"{stem}.txt", w, h)
    pseudo_items = parse_yolo_seg(Path(resolved["pseudo_labels_dir"]) / f"{stem}.txt", w, h)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    result = model.predict(source=img_bgr, conf=conf, iou=iou, imgsz=IMAGE_SIZE,
                           max_det=MAX_DET, retina_masks=RETINA_MASKS,
                           device=DEVICE_RESOLVED, verbose=False, save=False)[0]
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    inference_ms = (time.perf_counter() - t0) * 1000.0
    pred_items = predict_polygons(result)

    gt_masks = instance_masks(gt_items, w, h)
    pr_masks = instance_masks([(c, p) for c, _, p in pred_items], w, h)
    stats = {}
    for region in REGIONS:
        pairs = greedy_match(gt_masks[region], pr_masks[region])
        ious = [0.0] * len(gt_masks[region])
        for gi, _, val in pairs:
            ious[gi] = val
        stats[region] = {
            "n_gt": len(gt_masks[region]),
            "n_pred": len(pr_masks[region]),
            "n_matched": sum(1 for v in ious if v >= MATCH_IOU_THRESHOLD),
            "mean_iou": float(np.mean(ious)) if ious else np.nan,
            "max_iou": float(max(ious)) if ious else np.nan,
        }
    return img_bgr, gt_items, pseudo_items, pred_items, stats, inference_ms


def render_preview(img_bgr, gt_items, pseudo_items, pred_items, *, show_pred_masks,
                   show_pred_contours, show_pred_boxes, show_pred_conf,
                   show_gt, show_gt_fill, show_pseudo):
    """Compose the annotated RGB image from predictions, true GT and pseudo GT."""
    base = img_bgr.copy()
    overlay = base.copy()

    if show_gt_fill:
        for cls, poly in gt_items:
            pts = np.asarray(poly, dtype=np.int32).reshape(-1, 1, 2)
            cv2.fillPoly(overlay, [pts], _gt_bgr(cls))
    if show_pred_masks:
        for cls, _, poly in pred_items:
            pts = np.asarray(poly, dtype=np.int32).reshape(-1, 1, 2)
            cv2.fillPoly(overlay, [pts], _pred_bgr(cls))
    blended = cv2.addWeighted(overlay, WIDGET_MASK_ALPHA, base, 1 - WIDGET_MASK_ALPHA, 0)

    if show_pseudo:
        for cls, poly in pseudo_items:
            draw_dashed_polygon(blended, poly, _pseudo_bgr(cls), thickness=2,
                                dash_px=WIDGET_PSEUDO_DASH_PX, gap_px=WIDGET_PSEUDO_GAP_PX)
    if show_gt:
        for cls, poly in gt_items:
            draw_dashed_polygon(blended, poly, _gt_bgr(cls), thickness=2,
                                dash_px=WIDGET_GT_DASH_PX, gap_px=WIDGET_GT_GAP_PX)

    for cls, conf, poly in pred_items:
        color = _pred_bgr(cls)
        pts = np.asarray(poly, dtype=np.int32).reshape(-1, 1, 2)
        if show_pred_contours:
            cv2.polylines(blended, [pts], True, color, WIDGET_CONTOUR_THICK, cv2.LINE_AA)
        xs, ys = np.asarray(poly)[:, 0], np.asarray(poly)[:, 1]
        x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
        if show_pred_boxes:
            cv2.rectangle(blended, (x1, y1), (x2, y2), color, WIDGET_BOX_THICKNESS)
        if show_pred_conf:
            region = CLASS_NAMES.get(int(cls), str(cls))
            draw_label_chip(blended, (x1, y1), f"{region} {conf:.2f}", color)

    return cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)


dd_dataset = widgets.Dropdown(
    options=[(DATASETS[k]["display"], k) for k in ACTIVE_DATASETS],
    description="Dataset:", layout=widgets.Layout(width="320px"))
dd_image = widgets.Dropdown(description="Object:", layout=widgets.Layout(width="560px"))

btn_show = widgets.Button(description="Show", button_style="primary", icon="eye",
                          layout=widgets.Layout(width="110px"))
btn_prev = widgets.Button(description="Prev", icon="arrow-left",
                          layout=widgets.Layout(width="110px"))
btn_next = widgets.Button(description="Next", icon="arrow-right",
                          layout=widgets.Layout(width="110px"))
btn_rand = widgets.Button(description="Random", button_style="info", icon="random",
                          layout=widgets.Layout(width="120px"))
btn_save = widgets.Button(description="Save PNG", icon="save",
                          layout=widgets.Layout(width="120px"))

conf_slider = widgets.FloatSlider(
    value=CONF_THRESHOLD, min=0.0, max=1.0, step=0.01,
    description="Conf:", readout_format=".2f", layout=widgets.Layout(width="360px"))
iou_slider = widgets.FloatSlider(
    value=IOU_THRESHOLD, min=0.10, max=0.90, step=0.05,
    description="IoU:", readout_format=".2f", layout=widgets.Layout(width="360px"))

cb_pred_masks    = widgets.Checkbox(value=True,  description="Pred Masks", indent=False)
cb_pred_contours = widgets.Checkbox(value=True,  description="Pred Polygons", indent=False)
cb_pred_boxes    = widgets.Checkbox(value=False, description="Pred Boxes", indent=False)
cb_pred_conf     = widgets.Checkbox(value=True,  description="Pred Labels", indent=False)
cb_gt            = widgets.Checkbox(value=True,  description="GT Dashed", indent=False)
cb_gt_fill       = widgets.Checkbox(value=False, description="GT Masks", indent=False)
cb_pseudo        = widgets.Checkbox(value=False, description="Pseudo GT Dashed", indent=False)

CHECKBOXES = [cb_pred_masks, cb_pred_contours, cb_pred_boxes, cb_pred_conf,
              cb_gt, cb_gt_fill, cb_pseudo]

out_preview = widgets.Output()
_preview_state = {"stems": []}


def _refresh_images(*_):
    key = dd_dataset.value
    _preview_state["stems"] = TRUE_GT[key]["accepted"] if key else []
    dd_image.options = [(s, s) for s in _preview_state["stems"]]
    if dd_image.options:
        dd_image.value = dd_image.options[0][1]


def _render(save=False):
    with out_preview:
        clear_output(wait=True)
        if not dd_dataset.value or not dd_image.value:
            print("no dataset with accepted true-GT objects is available")
            return
        resolved = DATASETS[dd_dataset.value]
        stem = dd_image.value
        img_path = image_path_for(resolved["images_dir"], stem)
        if img_path is None:
            print(f"image not found for {stem}")
            return

        (img_bgr, gt_items, pseudo_items, pred_items,
         stats, inference_ms) = preview_predict(resolved, img_path,
                                                conf_slider.value, iou_slider.value)
        rgb = render_preview(
            img_bgr, gt_items, pseudo_items, pred_items,
            show_pred_masks=cb_pred_masks.value,
            show_pred_contours=cb_pred_contours.value,
            show_pred_boxes=cb_pred_boxes.value,
            show_pred_conf=cb_pred_conf.value,
            show_gt=cb_gt.value, show_gt_fill=cb_gt_fill.value,
            show_pseudo=cb_pseudo.value)

        h, w = rgb.shape[:2]
        fig_h = WIDGET_FIG_W * h / w + WIDGET_TITLE_PAD_IN
        fig, ax = plt.subplots(figsize=(WIDGET_FIG_W, fig_h))
        ax.imshow(rgb)
        ax.axis("off")
        device_tag = (f"GPU ({torch.cuda.get_device_name(0)})"
                      if torch.cuda.is_available() else "CPU")
        ax.set_title(
            f"{resolved['display']}  |  {stem}  |  {w}x{h}  |  "
            f"pred={len(pred_items)}  conf>={conf_slider.value:.2f}  |  "
            f"inference={inference_ms:.1f} ms ({device_tag})\n"
            f"solid=prediction   dashed=true GT   short-dashed=pseudo GT   |   "
            f"body=red  cap=green  label=blue",
            fontsize=10)
        fig.tight_layout()

        save_path = None
        if save:
            save_path = (RESULTS_DIR /
                         f"{PROJECT_NAME}_preview_{resolved['key']}_{stem}.png")
            fig.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()

        print(f"{'region':<7}{'n GT':>6}{'n Pred':>8}{'matched':>9}"
              f"{'mean IoU':>11}{'max IoU':>10}")
        print("-" * 51)
        for region in REGIONS:
            s = stats[region]
            mean_s = "n/a" if np.isnan(s["mean_iou"]) else f"{s['mean_iou']:.3f}"
            max_s  = "n/a" if np.isnan(s["max_iou"]) else f"{s['max_iou']:.3f}"
            print(f"{region:<7}{s['n_gt']:>6}{s['n_pred']:>8}{s['n_matched']:>9}"
                  f"{mean_s:>11}{max_s:>10}")
        if save_path is not None:
            print(f"\nsaved: {save_path}")


def _step(delta):
    opts = [v for _, v in dd_image.options]
    if not opts:
        return
    idx = (opts.index(dd_image.value) + delta) % len(opts)
    dd_image.value = opts[idx]


dd_dataset.observe(lambda ch: (_refresh_images(), _render())
                   if ch["name"] == "value" else None, names="value")
dd_image.observe(lambda ch: _render() if ch["name"] == "value" else None, names="value")
for cb in CHECKBOXES:
    cb.observe(lambda ch: _render() if ch["name"] == "value" else None, names="value")
btn_show.on_click(lambda _: _render())
btn_prev.on_click(lambda _: _step(-1))
btn_next.on_click(lambda _: _step(+1))
btn_rand.on_click(lambda _: dd_image.set_trait(
    "value", random.choice([v for _, v in dd_image.options])))
btn_save.on_click(lambda _: _render(save=True))

_refresh_images()
display(widgets.VBox([
    widgets.HBox([dd_dataset, btn_show, btn_prev, btn_next, btn_rand, btn_save]),
    dd_image,
    widgets.HBox([conf_slider, iou_slider]),
    widgets.HBox(CHECKBOXES),
    out_preview,
]))
_render()

In [ ]:
# === Cell 9: Aggregated Result Plots ===
# Standalone: re-imports its dependencies and reads the saved result files.
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

PLOT_PROJECT_NAME = "20260824_YOLO_Generalizability_Benchmark"
PLOT_RESULTS_DIR  = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/08_Generalizability/results"
                         ) / PLOT_PROJECT_NAME

_PFW         = 16 / 2.54
_PFONT_SIZE  = 10
_PBAND_COLOR = "#f0f0f0"
PLOT_REGIONS = ["body", "cap", "label"]
PLOT_REGION_COLORS = {"body": "#FF0000", "cap": "#00FF00", "label": "#0000FF"}
PLOT_POOLED_COLOR = "#7f7f7f"
PLOT_TASK = "seg"

plt.rcParams["font.family"]     = "DejaVu Sans"
plt.rcParams["font.size"]       = _PFONT_SIZE
plt.rcParams["axes.titlesize"]  = _PFONT_SIZE + 1
plt.rcParams["axes.labelsize"]  = _PFONT_SIZE
plt.rcParams["legend.fontsize"] = _PFONT_SIZE - 1
plt.rcParams["savefig.dpi"]     = 150


def _p_panel_label(ax, letter, title):
    ax.set_title(f"$\\bf{{({letter})}}$ {title}")


def _p_add_bands(ax, n):
    for i in range(n):
        if i % 2 == 0:
            ax.axvspan(i - 0.5, i + 0.5, color=_PBAND_COLOR, zorder=0)


metric_files = sorted(PLOT_RESULTS_DIR.glob(f"{PLOT_PROJECT_NAME}_*_metrics.json"))
metric_files = [p for p in metric_files if not p.name.endswith("_aggregate_metrics.json")]
metrics_by_ds = {}
for path in metric_files:
    data = json.loads(path.read_text(encoding="utf-8"))
    metrics_by_ds[data["dataset"]] = data
aggregate = json.loads(
    (PLOT_RESULTS_DIR / f"{PLOT_PROJECT_NAME}_aggregate_metrics.json").read_text(
        encoding="utf-8"))
per_image = pd.read_csv(PLOT_RESULTS_DIR / f"{PLOT_PROJECT_NAME}_per_image_all.csv")

PLOT_DS_ORDER  = ["spectralwaste", "warp", "packwise"]
PLOT_DS_COLORS = {"spectralwaste": "#1f77b4", "warp": "#ff7f0e", "packwise": "#2ca02c"}
_FALLBACK_COLORS = ["#d62728", "#9467bd", "#8c564b", "#e377c2"]

DS_ORDER = ([k for k in PLOT_DS_ORDER if k in metrics_by_ds]
            + [k for k in metrics_by_ds if k not in PLOT_DS_ORDER])
DS_LABEL = {k: metrics_by_ds[k]["display"] for k in DS_ORDER}
DS_COLOR = {k: PLOT_DS_COLORS.get(k, _FALLBACK_COLORS[i % len(_FALLBACK_COLORS)])
            for i, k in enumerate(DS_ORDER)}
SERIES    = DS_ORDER + ["pooled"]
SERIES_LABEL = {**DS_LABEL, "pooled": "Pooled"}
SERIES_COLOR = {**DS_COLOR, "pooled": PLOT_POOLED_COLOR}

METRIC_FIELDS = [("mAP50", "map50", "map_pass"),
                 ("mAP50-95", "map50_95", "map_pass"),
                 ("Precision", "precision", "pr_pass"),
                 ("Recall", "recall", "pr_pass"),
                 ("F1", "f1", "pr_pass")]


def _series_value(series_key, field, block):
    if series_key == "pooled":
        return float(aggregate["ultralytics"][PLOT_TASK][field])
    return float(metrics_by_ds[series_key]["ultralytics"][block][PLOT_TASK][field])


# Figure 1 — overall metrics per dataset plus the pooled column.
fig, ax = plt.subplots(figsize=(_PFW, _PFW * 0.55))
n_groups = len(METRIC_FIELDS)
_p_add_bands(ax, n_groups)
width = 0.8 / len(SERIES)
xs = np.arange(n_groups)
max_val = 0.0
for si, key in enumerate(SERIES):
    offs = (si - (len(SERIES) - 1) / 2) * width
    vals = [_series_value(key, field, block) for _, field, block in METRIC_FIELDS]
    max_val = max(max_val, max(vals))
    ax.bar(xs + offs, vals, width=width, color=SERIES_COLOR[key], alpha=0.85,
           edgecolor="black", linewidth=0.3, label=SERIES_LABEL[key], zorder=2)
    for x, v in zip(xs + offs, vals):
        ax.text(x, v + max(0.005, 0.012 * max(max_val, 1e-6)), f"{v:.2f}",
                ha="center", va="bottom", fontweight="bold",
                fontsize=_PFONT_SIZE - 4, rotation=90, zorder=3)
ax.set_xticks(xs)
ax.set_xticklabels([n for n, _, _ in METRIC_FIELDS])
ax.set_xlim(-0.5, n_groups - 0.5)
ax.set_ylabel("Score")
ax.set_ylim(0, max(max_val, 0.05) * 1.18)
ax.set_title("Deployment YOLO Generalizability Across Third Party Datasets")
ax.legend(loc="upper right", ncols=1)
fig.savefig(PLOT_RESULTS_DIR / f"{PLOT_PROJECT_NAME}_overall_metrics.png",
            dpi=150, bbox_inches="tight")
plt.show()


# Figure 2 — per region mAP50-95 and F1 by dataset.
fig, axes = plt.subplots(1, 2, figsize=(_PFW, _PFW * 0.55))
panel_specs = [("a", "Per Region mAP50-95", "map50_95", "map_pass"),
               ("b", "Per Region F1", "f1", "pr_pass")]
for ax, (letter, title, field, block) in zip(axes, panel_specs):
    _p_add_bands(ax, len(PLOT_REGIONS))
    xs = np.arange(len(PLOT_REGIONS))
    width = 0.8 / len(DS_ORDER)
    max_val = 0.0
    bars = []
    for si, key in enumerate(DS_ORDER):
        offs = (si - (len(DS_ORDER) - 1) / 2) * width
        vals = [float(metrics_by_ds[key]["ultralytics"][block][PLOT_TASK]["per_class"]
                      .get(r, {}).get(field, 0.0)) for r in PLOT_REGIONS]
        max_val = max(max_val, max(vals))
        ax.bar(xs + offs, vals, width=width, color=DS_COLOR[key], alpha=0.85,
               edgecolor="black", linewidth=0.3, label=DS_LABEL[key], zorder=2)
        bars.append((xs + offs, vals))
    for positions, vals in bars:
        for x, v in zip(positions, vals):
            ax.text(x, v + max(0.005, 0.012 * max(max_val, 1e-6)), f"{v:.2f}",
                    ha="center", va="bottom", fontweight="bold",
                    fontsize=_PFONT_SIZE - 4, rotation=90, zorder=3)
    ax.set_xticks(xs)
    ax.set_xticklabels([r.title() for r in PLOT_REGIONS])
    ax.set_xlim(-0.5, len(PLOT_REGIONS) - 0.5)
    ax.set_ylim(0, max(max_val, 0.05) * 1.18)
    ax.set_ylabel("Score")
    _p_panel_label(ax, letter, title)
axes[0].legend(loc="upper right", ncols=1)
fig.tight_layout()
fig.savefig(PLOT_RESULTS_DIR / f"{PLOT_PROJECT_NAME}_per_region_metrics.png",
            dpi=150, bbox_inches="tight")
plt.show()


# Figure 3 — per image IoU distributions from the greedy matching pass.
fig, ax = plt.subplots(figsize=(_PFW, _PFW * 0.55))
positions, data, colors, ticks = [], [], [], []
width = 0.8 / len(PLOT_REGIONS)
for di, key in enumerate(DS_ORDER):
    sub = per_image[per_image["dataset"] == key]
    for ri, region in enumerate(PLOT_REGIONS):
        vals = sub[f"{region}_mean_iou_all_gt"].dropna().to_numpy()
        positions.append(di + (ri - (len(PLOT_REGIONS) - 1) / 2) * width)
        data.append(vals if vals.size else np.array([np.nan]))
        colors.append(PLOT_REGION_COLORS[region])
    ticks.append(di)
bp = ax.boxplot(data, positions=positions, widths=width * 0.85, patch_artist=True,
                showfliers=False, showmeans=True, meanline=True,
                medianprops=dict(color="black", lw=1.3, linestyle="-"),
                meanprops=dict(color="black", lw=1.1, linestyle="--"))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color); patch.set_alpha(0.85)
    patch.set_edgecolor("black"); patch.set_linewidth(0.3)
ax.set_xticks(ticks)
ax.set_xticklabels([DS_LABEL[k] for k in DS_ORDER])
ax.set_xlim(-0.5, len(DS_ORDER) - 0.5)
ax.set_ylim(0, 1.0 * 1.18)
ax.set_ylabel("Per Image Mean IoU")
ax.set_title("Per Region Segmentation IoU By Dataset")
handles = [plt.Rectangle((0, 0), 1, 1, facecolor=PLOT_REGION_COLORS[r], alpha=0.85,
                         edgecolor="black", linewidth=0.3, label=r.title())
           for r in PLOT_REGIONS]
handles += [Line2D([0], [0], color="black", lw=1.3, linestyle="-", label="Median"),
            Line2D([0], [0], color="black", lw=1.1, linestyle="--", label="Mean")]
ax.legend(handles=handles, loc="upper right", ncols=1)
fig.savefig(PLOT_RESULTS_DIR / f"{PLOT_PROJECT_NAME}_iou_distributions.png",
            dpi=150, bbox_inches="tight")
plt.show()


# Printed only; the saved metrics table is written by the Save Result Tables cell.
table = pd.DataFrame([{
    "Dataset": SERIES_LABEL[k],
    "Images": (aggregate["n_images"] if k == "pooled" else metrics_by_ds[k]["n_images"]),
    **{name: _series_value(k, field, block) for name, field, block in METRIC_FIELDS},
    "Micro F1 (Match)": (aggregate if k == "pooled" else metrics_by_ds[k]
                         )["greedy_matching"]["micro"]["f1"],
} for k in SERIES])
print(table.to_string(index=False, float_format=lambda v: f"{v:.3f}"))